# From Linear Layers to Convolutional Neural Networks - CIFAR-10 Exercises

**Objective.**  
In this notebook, you will implement and compare two neural models on **CIFAR-10**:

1. a `LinearModel`, which treats each image as a flat vector;
2. a `CNNModel`, which keeps the spatial and color structure of the image.

CIFAR-10 is more difficult than handwritten digit datasets because the images are natural color images with larger intra-class variability.

The notebook provides the dataset loading, training loop, and evaluation tools.  
Your work is to complete the model architectures and analyze the results.

The general learning loop is:

$$
\text{prediction} \rightarrow \text{loss} \rightarrow \text{zero grad} \rightarrow \text{backward} \rightarrow \text{step}
$$


## Preamble - Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Optional GPU information in Google Colab
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Dataset - CIFAR-10

We use the CIFAR-10 dataset from `torchvision.datasets`.

Each image is a color image:

$$
X \in \mathbb{R}^{3 \times 32 \times 32}
$$

The objective is to predict one of 10 classes:

$$
y \in \{0, 1, 2, \ldots, 9\}
$$

The classes are:

`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`.

This dataset is useful for this exercise because:

- it is free and well-known;
- it is more realistic than handwritten digits;
- it is still small enough to run in Google Colab;
- it makes the difference between linear layers and convolutional layers more visible.


In [ ]:
# CIFAR-10 normalization constants commonly used with torchvision examples
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616),
    ),
])

# download=True automatically downloads the dataset if it is not already present
train_dataset_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_dataset_full = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

class_names = train_dataset_full.classes
print("Classes:", class_names)
print("Full train set:", len(train_dataset_full))
print("Full test set:", len(test_dataset_full))

## Working Subset

CIFAR-10 contains 50,000 training images and 10,000 test images.

To keep the exercise fast in class, we use a subset by default.  
You can set `USE_SUBSET = False` to train on the full dataset.

In [ ]:
USE_SUBSET = True
N_TRAIN = 10000
N_TEST = 2000
BATCH_SIZE = 128

if USE_SUBSET:
    train_indices = np.random.choice(len(train_dataset_full), N_TRAIN, replace=False)
    test_indices = np.random.choice(len(test_dataset_full), N_TEST, replace=False)
    train_dataset = Subset(train_dataset_full, train_indices)
    test_dataset = Subset(test_dataset_full, test_indices)
else:
    train_dataset = train_dataset_full
    test_dataset = test_dataset_full

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Train samples used:", len(train_dataset))
print("Test samples used:", len(test_dataset))

## Visualize Some Images

In [ ]:
# Denormalization for visualization only
def denormalize(img):
    mean = torch.tensor((0.4914, 0.4822, 0.4465)).view(3, 1, 1)
    std = torch.tensor((0.2470, 0.2435, 0.2616)).view(3, 1, 1)
    return img * std + mean

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(10):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Input Representations

For the linear model, images must be flattened:

$$
3 \times 32 \times 32 \rightarrow 3072
$$

For the CNN model, images keep their structure:

$$
3 \times 32 \times 32
$$

The training function will handle both cases using the argument `flatten_input`.


## Helper Functions

This cell implements several helper functions. Make sure to read and understand them carefully, as they will be useful throughout the rest of the notebook.

In [ ]:
def count_parameters(model):
    # Counts the number of trainable parameters in the model.
    # Only parameters with requires_grad=True are included.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def prepare_inputs(x, flatten_input=False):
    # Moves the input batch to the selected device: CPU or GPU.
    x = x.to(device)

    # If needed, flattens each image into a 1D vector. Should be True for LinearModel and False for CNN
    # Example: (N, 3, 32, 32) becomes (N, 3072).
    if flatten_input:
        x = x.view(x.size(0), -1)

    return x


def train_model(model, train_loader, test_loader, lr=1e-3, epochs=5, flatten_input=False):
    # Moves the model to the selected device.
    model = model.to(device)

    # CrossEntropyLoss is commonly used for multi-class classification.
    criterion = nn.CrossEntropyLoss()

    # Adam updates the model parameters using the computed gradients.
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Stores the training loss, training accuracy, and test accuracy after each epoch.
    history = {"train_loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(epochs):
        # Enables training mode.
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for x, y in train_loader:
            # Prepares input images and labels.
            x = prepare_inputs(x, flatten_input=flatten_input)
            y = y.to(device)

            # Forward pass: computes class scores.
            logits = model(x)

            # Computes the classification loss.
            loss = criterion(logits, y)

            # Clears previous gradients.
            optimizer.zero_grad()

            # Backpropagation: computes gradients.
            loss.backward()

            # Updates model parameters.
            optimizer.step()

            # Accumulates the total loss over the batch.
            running_loss += loss.item() * x.size(0)

            # Converts logits into predicted class indices.
            preds = torch.argmax(logits, dim=1)

            # Counts correct predictions.
            correct += (preds == y).sum().item()
            total += y.size(0)

        # Computes average loss and accuracy for the epoch.
        train_loss = running_loss / total
        train_acc = correct / total

        # Evaluates the model on the test set.
        test_acc, _ = evaluate_model(model, test_loader, flatten_input=flatten_input)

        # Saves metrics for plotting.
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch {epoch + 1:03d}/{epochs:03d} | "
            f"loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"test_acc={test_acc:.4f}"
        )

    return history


def evaluate_model(model, data_loader, flatten_input=False):
    # Enables evaluation mode.
    model.eval()

    all_preds = []
    all_targets = []

    # Disables gradient computation to reduce memory usage and speed up evaluation.
    with torch.no_grad():
        for x, y in data_loader:
            # Prepares input images and labels.
            x = prepare_inputs(x, flatten_input=flatten_input)
            y = y.to(device)

            # Forward pass: computes class scores.
            logits = model(x)

            # Selects the class with the highest score.
            preds = torch.argmax(logits, dim=1)

            # Moves predictions and targets back to CPU for metric computation.
            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenates all batches into single arrays.
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    # Computes classification accuracy.
    acc = (all_preds == all_targets).mean()

    return acc, all_preds


def plot_history(history, title="Training history"):
    # Creates a figure with two plots: loss and accuracy.
    plt.figure(figsize=(10, 4))

    # Plots the training loss over epochs.
    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"])
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Train loss")

    # Plots training and test accuracy over epochs.
    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="Train accuracy")
    plt.plot(history["test_acc"], label="Test accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy")
    plt.legend()

    # Adds a global title and adjusts the layout.
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Exercise 1 - Implement and Train a Linear Model

In this first exercise, the image is treated as a vector.

$$
X \in \mathbb{R}^{3 \times 32 \times 32} \rightarrow x \in \mathbb{R}^{3072}
$$

The model receives 3072 input features and predicts 10 class scores.


## Question 1 - Implement `LinearModel`

Complete the class below.

Requirements:

- inherit from `nn.Module`;
- define one linear layer from 3072 input features to 10 output classes;
- implement the `forward` function;

In [ ]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: define a linear layer from 3072 input features to 10 classes
        self.linear = None

    def forward(self, x):
        # TODO: compute the logits
        # x shape: (N, 3072)
        logits = None
        
        return logits

## Question 2 - Train the Linear Model

Instructions:

- instantiate `LinearModel`;
- print the number of trainable parameters;
- plot the training history;
- evaluate the final model on the test set.

In [ ]:
# TODO: instantiate the model

# TODO: print the number of trainable parameters (sum(p.numel() for p in model.parameters())

# TODO: train the model with flatten_input=True

# TODO: plot the training history

# TODO: evaluate the final model on the test set

## Answer - Question 2

- What is the final training accuracy?
- What is the final test accuracy?
- Is there a large difference between train and test accuracy?
- Does the loss decrease smoothly?
- Is this performance satisfying for CIFAR-10?

**Your answer:**

TODO: write your answer here.

## Question 3 - Analyze the Errors

Display the confusion matrix of the linear model.

Analyze:

1. Which classes are often confused?
2. Are the errors visually understandable?
3. What limitation of the linear model could explain these errors?

In [ ]:
# TODO: compute predictions on the test set

# TODO: compute the confusion matrix

# TODO: plot the confusion matrix with class names

## Answer - Question 3

- Which classes are confused?
- Why can a linear model be limited on color image data?
- What spatial information is lost when the image is flattened?

**Your answer:**

TODO: write your answer here.

## Question 4 - Make the Linear Model Deeper

Create a deeper linear model with at least one hidden layer.

Instructions:

- add at least one hidden linear layer;
- add a non-linear activation function between two linear layers;
- train the model with the same protocol as the simple linear model;
- compare the performance, training behavior, and number of parameters.

Analyze:

1. Are the performances better?
2. Does the model train faster or slower?
3. Does the model have more parameters?
4. Do you observe overfitting?

In [ ]:
class DeepLinearModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: define a deeper linear architecture
        # TODO: include at least one hidden layer
        # TODO: include at least one activation function
        # TODO: end with 10 output classes

    def forward(self, x):
        # TODO: return the logits
        logits = None
       
        return logits


# TODO: instantiate, train, evaluate, and compare with LinearModel

## Answer - Question 4

- Did the deeper model improve test accuracy?
- Did it increase the number of parameters?
- Did it make training more stable, less stable, faster, or slower?

**Your answer:**

TODO: write your answer here.

# Exercise 2 - Implement and Train a CNN Model

In this second exercise, the image is kept as a structured object.

The input shape is:

$$
X \in \mathbb{R}^{3 \times 32 \times 32}
$$

A convolutional model can exploit local spatial patterns such as edges, textures, and shapes.


## Question 1 - Implement `CNNModel`

Complete the class below.

Requirements:

- use one convolution layer;
- use one ReLU activation;
- use one pooling operation to reduce the spatial resolution;
- flatten the feature maps;
- use one final linear layer for the 10 classes.

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: define a convolution layer
        # input channels = 3, output channels = 16, kernel size = 3, padding = 'same'
        self.conv = None

        # objective: reduce 32x32 images to 16x16 feature maps
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # Pooling reduces the spatial size of feature maps -> https://docs.pytorch.org/docs/2.12/generated/torch.nn.MaxPool2d.html

        # TODO: define the final linear layer
        # after convolution and pooling, compute the flattened size carefully
        self.fc = None

    def forward(self, x):
        # x shape: (N, 3, 32, 32)

        # TODO: convolution
        x = None

        # TODO: activation
        x = None

       # TODO: pooling
        x = self.pool(x)

        # TODO: flatten
        x = torch.flatten(x, start_dim=1) # flatten converts the 2D feature maps into a 1D vector that can be given to the final linear layer.

        # TODO: final linear layer
        
        logits = None
        
        return logits

## Question 2 - Train the CNN Model

Instructions:

- instantiate `CNNModel`;
- print the number of trainable parameters;
- plot the training history;
- evaluate the final model on the test set.

In [ ]:
# TODO: instantiate the CNN model

# TODO: print the number of trainable parameters

# TODO: plot the training history

# TODO: evaluate the final CNN on the test set

## Answer - Question 2

- What is the final training accuracy?
- What is the final test accuracy?
- Is the CNN better than the linear model?
- Is the CNN training curve more stable, less stable, or similar?

**Your answer:**

TODO: write your answer here.

## Question 3 - Compare Linear Model and CNN Model

Complete the comparison table.

You should compare:

- number of trainable parameters;
- train accuracy;
- test accuracy;
- qualitative behavior during training.

In [ ]:
# TODO: fill this dictionary with your results
results = {
    "Model": ["LinearModel", "CNNModel"],
    "Parameters": [None, None],
    "Train accuracy": [None, None],
    "Test accuracy": [None, None],
}

results

## Answer - Question 3

- Which model is more accurate?
- Which model is heavier in terms of parameters?
- Is the heavier model necessarily the best model?
- Why can a CNN be efficient for images even with a small number of parameters?

**Your answer:**

TODO: write your answer here.

## Question 4 - Make the CNN Deeper

Create a deeper CNN.

Instructions:

- add at least one additional convolutional layer;
- add activation functions after convolutional layers;
- flatten the feature maps before the final classification layer;
- ensure that the final layer predicts 10 classes;
- train and evaluate the model with the same protocol as the first CNN.

Analyze:

1. Are the performances better?
2. Is the model heavier?
3. Does the model converge faster or slower?
4. Is there a risk of overfitting?

In [ ]:
class DeepCNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: define a deeper CNN architecture
        # TODO: include at least two convolutional layers
        # TODO: include activation functions
        # TODO: include pooling if needed
        # TODO: include a final classification layer with 10 outputs

    def forward(self, x):
        # TODO: return the logits
        logits = None
       
        return logits


# TODO: instantiate, train, evaluate, and compare with CNNModel

## Answer - Question 4

- Did the deeper CNN improve test accuracy?
- Did the deeper CNN improve train accuracy only?
- What would you try next to improve generalization?

**Your answer:**

TODO: write your answer here.

# Final Synthesis

Write a short conclusion comparing the models.

Your conclusion should answer:

1. What is the main limitation of a linear model on natural images?
2. Why are convolutions useful for image data?
3. Which model had the best test accuracy?
4. Which model had the most parameters?
5. Did making the models deeper always improve the results?

**Your conclusion:**

...

# Optional Extension - Suggested Experiments

You can try:

- changing the learning rate;
- changing the number of epochs;
- changing the batch size;
- training on the full CIFAR-10 dataset;
- adding one hidden layer to the linear model;
- adding a second convolution layer;
- adding `BatchNorm2d` after convolution layers;
- adding dropout;
- feel free to add whatever you want to improve the performance